In [1]:
from collections import defaultdict
from pathlib import Path

import pandas as pd
from acteval.plot.frequency import plot_agg_acts
from acteval.plot.times import joint_time_distributions_plot
from matplotlib import pyplot as plt
from matplotlib.patches import Patch
from plots import plot_b

In [40]:
cmap = {
    "home": (0.5529411764705883, 0.8274509803921568, 0.7803921568627451),
    "other": (1.0, 1.0, 0.7019607843137254),
    "shop": (0.7450980392156863, 0.7294117647058823, 0.8549019607843137),
    "work": (0.984313725490196, 0.5019607843137255, 0.4470588235294118),
    "escort": (0.5019607843137255, 0.6941176470588235, 0.8274509803921568),
    "visit": (0.9921568627450981, 0.7058823529411765, 0.3843137254901961),
    "education": (0.7019607843137254, 0.8705882352941177, 0.4117647058823529),
    "medical": (0.9882352941176471, 0.803921568627451, 0.8980392156862745),
    "travel": (0.8509803921568627, 0.8509803921568627, 0.8509803921568627),
    "leisure": (0.996078431372549, 0.8509803921568627, 0.6509803921568628),
}


def latest(path: Path):
    versions = sorted(
        [
            d
            for d in path.iterdir()
            if d.is_dir() and d.name.startswith("version")
        ]
    )
    print(
        f"Found versions for {path.name}: {[v.name for v in versions]}. Returning {versions[-1]}"
    )
    return path.name, versions[-1]


def earliest(path: Path):
    versions = sorted(
        [
            d
            for d in path.iterdir()
            if d.is_dir() and d.name.startswith("version")
        ]
    )
    print(
        f"Found versions for {path.name}: {[v.name for v in versions]}. Returning {versions[-1]}"
    )
    return path.name, versions[0]


def iter_models(path: Path):
    for dir in path.iterdir():

        if dir.is_dir():

            yield latest(dir)


def find_nth(path: Path, n=0):

    ranks = (
        pd.read_csv(path / "domain_distances.csv")
        .drop(columns=["domain", "observed", "mean", "std"], errors="ignore")
        .rank(axis=1, method="min")
        .sum(axis=0)
    )
    ranked = [i for _, i in sorted(zip(ranks, ranks.index))]
    top = ranked[n]
    dir = path / top
    yield latest(dir)


def best_quality(path: Path, n=0):

    data = pd.read_csv(path / "domain_distances.csv")
    data = data[data.domain == "sample quality"].drop(
        columns=["domain", "observed", "mean", "std"], errors="ignore"
    )
    ranks = data.rank(axis=1, method="min").sum(axis=0)
    ranked = [i for _, i in sorted(zip(ranks, ranks.index))]
    top = ranked[n]
    dir = path / top
    yield latest(dir)


def load_attributes(path: Path, default=None):
    if (path / "synthetic_attributes.csv").exists():
        print(f"Loading synthetic attributes from {path}")
        return pd.read_csv(path / "synthetic_attributes.csv")
    if (path / "synthetic_labels.csv").exists():
        print(f"Loading synthetic labels from {path}")
        return pd.read_csv(path / "synthetic_labels.csv")
    print(f"Failed to find attributes at {path}")
    return default


def split_on(schedules, attributes, by="work_status", remove_unknown=False):
    splits = attributes.groupby(by)
    split = {
        k: schedules.loc[schedules.pid.isin(list(v.pid))]
        for k, v in splits
        if k != "unknown" or not remove_unknown
    }
    print(
        f"Split schedules into {len(split)} groups by {by}: {list(split.keys())}"
    )
    return split


def styler(df, vmin=-1, vmax=1):
    df = -df
    df.loc["24", "london"] = None
    df.loc["00-24", "london"] = None
    for yrs in ["24", "23-24"]:
        for source in ["west", "global"]:
            df.loc[yrs, source] = None

    styled = df.style.background_gradient(
        cmap="RdYlGn", vmin=vmin, vmax=vmax
    ).format("{:.1%}", na_rep="")

    return styled

In [3]:
training_data_route = Path("../../tmp/foundation/world")

training_schedules = {}
training_attributes = {}

counts = defaultdict(list)
for scope in [
    "training",
    "london",
    "uk",
    "dutch",
    "us",
    "chicago",
    "queensland",
    "victoria",
    "kr",
]:
    schedules_path = Path(training_data_route / f"{scope}_activities.csv")
    attributes_path = Path(training_data_route / f"{scope}_attributes.csv")
    training_schedules[f"{scope}"] = pd.read_csv(schedules_path)
    training_attributes[f"{scope}"] = pd.read_csv(attributes_path)

/tmp/ipykernel_1500363/4206428320.py:21: DtypeWarning: Columns (27) have mixed types. Specify dtype option on import or set low_memory=False.
  training_attributes[f"{scope}"] = pd.read_csv(attributes_path)
/tmp/ipykernel_1500363/4206428320.py:21: DtypeWarning: Columns (27) have mixed types. Specify dtype option on import or set low_memory=False.
  training_attributes[f"{scope}"] = pd.read_csv(attributes_path)


In [4]:
sizes = {}
for name, atts in training_attributes.items():
    print(name, atts.pid.nunique(), len(atts))
    sizes[name] = len(atts)

training 2376675 2376675
london 27432 27432
uk 200755 200755
dutch 117902 117902
us 15309 15309
chicago 25626 25626
queensland 14749 14749
victoria 16515 16515
kr 120100 120100


In [5]:
def freq_plotter(
    target_schedules,
    target_attributes,
    schedules,
    attributes,
    by=None,
    filter=None,
    names=None,
):

    acts = set()
    for sched in schedules.values():
        acts.update(sched.act.unique())

    act_order = target_schedules.act.value_counts().index.to_list()
    class_map = {n: i for i, n in enumerate(act_order)}

    if by is None:
        by = "employment"
        FILTER = ["student", "employed", "unemployed", "retired", "unknown"]
        NAMES = ["Student", "Employed", "Unemployed", "Retired", "Unknown"]
    else:
        if filter is None:
            FILTER = set()
            for atts in attributes.values():
                FILTER.update(atts[by].unique().tolist())
            FILTER = list(FILTER)
        else:
            FILTER = filter
        if names is None:
            NAMES = [str(f).title() for f in FILTER]

    h = len(schedules)
    w = len(FILTER)
    fig, axes = plt.subplots(
        h, w + 2, figsize=(w * 2.2, h * 2), sharex=False, sharey=False
    )

    for i, (name, scheds) in enumerate(schedules.items()):

        splits = split_on(scheds, attributes[name], by=by)

        for j, f in enumerate(FILTER):
            axes[0, j].set_title(NAMES[j], fontsize=12)
            selected = splits.get(f)
            if selected is None:
                axes[i, j].axis("off")
                continue
            ax = plot_agg_acts(
                "",
                selected,
                class_map=class_map,
                ax=axes[i, j],
                legend=False,
                color=cmap,
            )
            ax.set_title(None)
            ax.set_ylabel(None)
            ax.set_xlabel(None)
            ax.set_yticks([])
            ax.set_xticks([])

            axes[-1, j].set_xticks(
                [i / 10 for i in [0, 240, 480, 720, 960, 1200, 1440]],
                labels=[
                    "",
                    "04:00",
                    "08:00",
                    "12:00",
                    "16:00",
                    "20:00",
                    "24:00",
                ],
                rotation=90,
                fontsize=8,
            )

        axes[0, -2].set_title("Combined", fontsize=12)
        ax = plot_agg_acts(
            "",
            scheds,
            class_map=class_map,
            ax=axes[i, -2],
            legend=False,
            color=cmap,
        )
        ax.set_title(None)
        ax.set_ylabel(None)
        ax.set_xlabel(None)
        ax.set_yticks([])
        ax.set_xticks([])

        axes[-1, -2].set_xticks(
            [i / 10 for i in [0, 240, 480, 720, 960, 1200, 1440]],
            labels=["", "04:00", "08:00", "12:00", "16:00", "20:00", "24:00"],
            rotation=90,
            fontsize=8,
        )

        axes[i, 0].set_ylabel(name, fontsize=12)
        axes[i, 0].set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
        axes[i, 0].set_yticklabels(
            ["", "20% ", "40% ", "60% ", "80% ", "100%"],
            rotation=90,
            fontsize=8,
        )

        axes[i, -1].axis("off")

    elements = [
        Patch(facecolor=cmap[act], label=act.title()) for act in act_order
    ]
    axes[0, -1].axis("off")
    axes[0, -1].legend(handles=elements, loc="center left", frameon=False)

    plt.subplots_adjust(wspace=0.04, hspace=0.05)

In [17]:
data_route = Path("../../logs/foundations/world")
batch_paths = {}
for scope in [
    "london",
    "kr",
    "dutch",
    "uk",
    "dutch",
    "us",
    "chicago",
    "queensland",
    "victoria",
]:
    for e in ["", "_base"]:
        batch_paths[f"{scope}{e}"] = data_route / f"{scope}{e}"

schedules = {
    n: pd.read_csv(latest(p)[1] / "synthetic_schedules.csv")
    for n, p in batch_paths.items()
}
attributes = {
    n: load_attributes(latest(p)[1], default=None)
    for n, p in batch_paths.items()
}

Found versions for london: ['version_0']. Returning ../../logs/foundations/world/london/version_0
Found versions for london_base: ['version_0']. Returning ../../logs/foundations/world/london_base/version_0
Found versions for kr: ['version_0', 'version_1', 'version_2']. Returning ../../logs/foundations/world/kr/version_2
Found versions for kr_base: ['version_0', 'version_1', 'version_2', 'version_3']. Returning ../../logs/foundations/world/kr_base/version_3
Found versions for dutch: ['version_0', 'version_1', 'version_2']. Returning ../../logs/foundations/world/dutch/version_2
Found versions for dutch_base: ['version_0', 'version_1', 'version_2']. Returning ../../logs/foundations/world/dutch_base/version_2
Found versions for uk: ['version_0', 'version_1']. Returning ../../logs/foundations/world/uk/version_1
Found versions for uk_base: ['version_0', 'version_1']. Returning ../../logs/foundations/world/uk_base/version_1
Found versions for us: ['version_0']. Returning ../../logs/foundation

In [7]:
domain_distances = pd.read_csv(
    data_route / "eval/domains/distances.csv"
).set_index("domain")

sources = [
    "london",
    "uk",
    "dutch",
    "us",
    "chicago",
    "queensland",
    "victoria",
    "kr",
]
cols = []
for source in sources:
    cols.append(f"{source}_base")
    cols.append(f"{source}")
domain_distances[cols]

,london_base,london,uk_base,uk,dutch_base,dutch,us_base,us,chicago_base,chicago,queensland_base,queensland,victoria_base,victoria,kr_base,kr
domain,,,,,,,,,,,,,,,,
creativity,0.225044,0.234104,0.243299,0.222562,0.137412,0.135975,0.217087,0.219570,0.112626,0.125524,0.141413,0.139209,0.151512,0.161625,0.177929,0.248172
feasibility,0.007218,0.006015,0.014809,0.014968,0.028040,0.035055,0.051473,0.009537,0.080855,0.028799,0.074514,0.015120,0.144172,0.030336,0.023489,0.035604
participations,0.214908,0.216030,0.189960,0.168268,0.125578,0.132183,0.177580,0.129834,0.398858,0.333764,0.153046,0.091845,0.330018,0.216177,0.162840,0.308389
timing,0.119300,0.054690,0.058180,0.046329,0.043900,0.031175,0.186454,0.045222,0.117097,0.058448,0.185561,0.028876,0.149917,0.036283,0.082566,0.098826
transitions,0.013905,0.005419,0.005829,0.004062,0.005782,0.003930,0.019309,0.004099,0.014093,0.007616,0.015418,0.004472,0.019508,0.006991,0.014055,0.013122


In [8]:
results = pd.DataFrame()

for source in sources:
    old = domain_distances[f"{source}_base"]
    new = domain_distances[f"{source}"]
    change = (new - old) / old
    results[source] = change

sizes_row = pd.Series(sizes, name="size").drop("training").to_frame().T
results = pd.concat([sizes_row, results], axis=0)


def styler(df):
    df.iloc[1:] = -df.iloc[1:]

    # subset that includes only the first row, formatted as integer
    size_subset = pd.IndexSlice[df.index[0], :]
    # subset that excludes the first row
    subset = pd.IndexSlice[df.index[1:], :]

    styled = (
        df.style.format("{:,.0f}", na_rep="", subset=size_subset)
        .background_gradient(cmap="RdYlGn", vmin=-1, vmax=1, subset=subset)
        .format("{:.1%}", na_rep="", subset=subset)
    )
    return styled


# sort cols by size
results = results[sizes_row.T.sort_values("size", ascending=False).index]

results = results.drop("creativity", axis=0)

results.columns = [
    "UK",
    "Korea",
    "Netherlands",
    "London",
    "Chicago",
    "Victoria",
    "US",
    "Queensland",
]

# manually rename index
results.index = [
    "Sample size",
    "Feasibility",
    "Participation distribution",
    "Timing distribution",
    "Transitions distribution",
]

# reorder rows
results = results.loc[
    [
        "Sample size",
        "Feasibility",
        "Participation distribution",
        "Transitions distribution",
        "Timing distribution",
    ]
]


styler(results)

,UK,Korea,Netherlands,London,Chicago,Victoria,US,Queensland
Sample size,"200,755","120,100","117,902","27,432","25,626","16,515","15,309","14,749"
Feasibility,-1.1%,-51.6%,-25.0%,16.7%,64.4%,79.0%,81.5%,79.7%
Participation distribution,11.4%,-89.4%,-5.3%,-0.5%,16.3%,34.5%,26.9%,40.0%
Transitions distribution,30.3%,6.6%,32.0%,61.0%,46.0%,64.2%,78.8%,71.0%
Timing distribution,20.4%,-19.7%,29.0%,54.2%,50.1%,75.8%,75.7%,84.4%


In [18]:
training_schedules.keys()

dict_keys(['training', 'london', 'uk', 'dutch', 'us', 'chicago', 'queensland', 'victoria', 'kr'])

In [19]:
schedules.keys()

dict_keys(['london', 'london_base', 'kr', 'kr_base', 'dutch', 'dutch_base', 'uk', 'uk_base', 'us', 'us_base', 'chicago', 'chicago_base', 'queensland', 'queensland_base', 'victoria', 'victoria_base'])

In [24]:
from acteval.evaluate import compare_many

selected_targets = [
    "london",
    "kr",
    "dutch",
    "uk",
    "us",
    "chicago",
    "queensland",
    "victoria",
]

selected = [
    "london_base",
    "kr_base",
    "dutch_base",
    "uk_base",
    "us_base",
    "chicago_base",
    "queensland_base",
    "victoria_base",
]

res_base = compare_many(
    schedules_a={k: training_schedules[k] for k in selected_targets},
    attributes_a={k: training_attributes[k] for k in selected_targets},
    schedules_b={k: schedules[k] for k in selected},
    attributes_b={k: attributes[k] for k in selected},
    split_on=[
        "source",
        "year",
        "age",
        "employment",
        "hh_income",
        "hh_zone",
        "day",
    ],
    progress=True,
)

target [features]                                   0% │                         │    0/10 [00:00]


target [features]       3-gram                     40% │━━━━━━━━━━               │    4/10 [00:00]

target [features]                                 100% │━━━━━━━━━━━━━━━━━━━━━━━━━│   10/10 [00:00]














target [features]                                 100% │━━━━━━━━━━━━━━━━━━━━━━━━━│   10/10 [00:20]




















































































































target [features]                                 100% │━━━━━━━━━━━━━━━━━━━━━━━━━│   10/10 [00:47]
target [splits]                                   100% │━━━━━━━━━━━━━━━━━━━━━━━━━│   29/29 [00:47]
london_base [features]                            100% │━━━━━━━━━━━━━━━━━━━━━━━━━│   10/10 [00:47]
london_base [splits]                              100% │━━━━━━━━━━━━━━━━━━━━━━━━━│   29/29 [00:47]
target [features]                               0% │                         │    0/10 [00:00]


target [features]                             100% │━━━━━━━━━━━━━━━━━━━━━━━━━│   10/10 [00:12]






















































































In [25]:
res_base.domains

/home/fred/Projects/acteval/src/acteval/_result_frame.py:101: RuntimeWarning: The values in the array are unorderable. Pass `sort=False` to suppress this warning.
  combined_w = (self.weights[col] + target_weights) / 2
/home/fred/Projects/acteval/src/acteval/_result_frame.py:103: RuntimeWarning: The values in the array are unorderable. Pass `sort=False` to suppress this warning.
  weighted = (self.values[col].mul(combined_w)).groupby(groupby).sum()
/home/fred/Projects/acteval/src/acteval/_result_frame.py:101: RuntimeWarning: The values in the array are unorderable. Pass `sort=False` to suppress this warning.
  combined_w = (self.weights[col] + target_weights) / 2
/home/fred/Projects/acteval/src/acteval/_result_frame.py:103: RuntimeWarning: The values in the array are unorderable. Pass `sort=False` to suppress this warning.
  weighted = (self.values[col].mul(combined_w)).groupby(groupby).sum()
/home/fred/Projects/acteval/src/acteval/_result_frame.py:101: RuntimeWarning: The values in th

ScheduleView [domains]  (.combined / .by_attribute / .by_category)

                london::london_base  kr::kr_base  dutch::dutch_base  uk::uk_base  us::us_base  chicago::chicago_base  queensland::queensland_base  victoria::victoria_base
domain                                                                                                                                                                    
creativity                 0.221836     0.177756           0.129836     0.229547     0.178740               0.090343                     0.116073                 0.127014
feasibility                0.007218     0.027246           0.027038     0.014413     0.044054               0.066863                     0.064994                 0.120858
participations             0.063145     0.119172           0.053889     0.057965     0.079612               0.104931                     0.107785                 0.136496
timing                     0.063785     0.068073           0.032377     0.035

In [23]:
res.domains

ScheduleView [domains]  (.combined / .by_attribute / .by_category)

                london::london    kr::kr  dutch::dutch    uk::uk    us::us  chicago::chicago  queensland::queensland  victoria::victoria
domain                                                                                                                                  
creativity            0.233200  0.217785      0.127349  0.208542  0.188551          0.108565                0.121159            0.141632
feasibility           0.006015  0.032498      0.033458  0.014534  0.007675          0.022962                0.013237            0.025698
participations        0.059453  0.185338      0.046555  0.031032  0.053133          0.067689                0.065692            0.082409
timing                0.038858  0.077430      0.020409  0.020730  0.027237          0.030828                0.038100            0.027722
transitions           0.006459  0.008171      0.003068  0.003038  0.006168          0.005949                0.

In [28]:
from acteval.results import combine

combined = combine({"base": res_base, "world": res})
combined.domains

/tmp/ipykernel_1500363/2932478845.py:3: UserWarning: combine(): row index differs from 'base' for source(s) ['world'] (e.g. different activities present in different targets). Aggregation weighting for non-first sources is approximate — see combine()'s docstring.
  combined = combine({"base": res_base, "world":res})
/home/fred/Projects/acteval/src/acteval/_result_frame.py:101: RuntimeWarning: The values in the array are unorderable. Pass `sort=False` to suppress this warning.
  combined_w = (self.weights[col] + target_weights) / 2
/home/fred/Projects/acteval/src/acteval/_result_frame.py:103: RuntimeWarning: The values in the array are unorderable. Pass `sort=False` to suppress this warning.
  weighted = (self.values[col].mul(combined_w)).groupby(groupby).sum()
/home/fred/Projects/acteval/src/acteval/_result_frame.py:101: RuntimeWarning: The values in the array are unorderable. Pass `sort=False` to suppress this warning.
  combined_w = (self.weights[col] + target_weights) / 2
/home/fred

ScheduleView [domains]  (.combined / .by_attribute / .by_category)

                base::london::london_base  base::kr::kr_base  base::dutch::dutch_base  base::uk::uk_base  base::us::us_base  base::chicago::chicago_base  base::queensland::queensland_base  base::victoria::victoria_base  world::london::london  world::kr::kr  world::dutch::dutch  world::uk::uk  world::us::us  world::chicago::chicago  world::queensland::queensland  world::victoria::victoria
domain                                                                                                                                                                                                                                                                                                                                                                                                
creativity                       0.221836           0.177756                 0.129836           0.229547           0.178740                     0.0903

In [30]:
sources

['london', 'uk', 'dutch', 'us', 'chicago', 'queensland', 'victoria', 'kr']

In [ ]:
results = pd.DataFrame()

for source in sources:
    old = combined.domains.combined.distances[f"base::{source}::{source}_base"]
    new = combined.domains.combined.distances[f"world::{source}::{source}"]
    change = (new - old) / old
    results[source] = change

sizes_row = pd.Series(sizes, name="size").drop("training").to_frame().T
results = pd.concat([sizes_row, results], axis=0)


# sort cols by size
results = results[sizes_row.T.sort_values("size", ascending=False).index]

results = results.drop("creativity", axis=0)

results.columns = [
    "UK",
    "Korea",
    "Netherlands",
    "London",
    "Chicago",
    "Victoria",
    "US",
    "Queensland",
]

# manually rename index
results.index = [
    "Base sample size",
    "Feasibility",
    "Participation distribution",
    "Timing distribution",
    "Transitions distribution",
]

# reorder rows
results = results.loc[
    [
        "Base sample size",
        # "Feasibility",
        "Participation distribution",
        "Transitions distribution",
        "Timing distribution",
    ]
]


def styler(df):
    df.iloc[1:] = -df.iloc[1:]

    # subset that includes only the first row, formatted as integer
    size_subset = pd.IndexSlice[df.index[0], :]
    # subset that excludes the first row
    subset = pd.IndexSlice[df.index[1:], :]

    styled = (
        df.style.format("{:,.0f}", na_rep="", subset=size_subset)
        .background_gradient(cmap="RdYlGn", vmin=-1, vmax=1, subset=subset)
        .format("{:.1%}", na_rep="", subset=subset)
    )
    return styled


average = (
    results.loc[
        [
            "Participation distribution",
            "Transitions distribution",
            "Timing distribution",
        ],
        :,
    ]
    .mean(axis=0)
    .to_frame(name="Average")
    .T
)

results = pd.concat([results, average], axis=0)
styler(results)

,UK,Korea,Netherlands,London,Chicago,Victoria,US,Queensland
Base sample size,"200,755","120,100","117,902","27,432","25,626","16,515","15,309","14,749"
Participation distribution,46.5%,-55.5%,13.6%,5.8%,35.5%,39.6%,33.3%,39.1%
Transitions distribution,34.9%,53.5%,28.6%,67.4%,67.0%,80.4%,73.7%,74.9%
Timing distribution,41.9%,-13.7%,37.0%,39.1%,42.0%,72.8%,74.1%,68.5%
Average,41.1%,-5.3%,26.4%,37.4%,48.2%,64.3%,60.3%,60.8%


In [49]:
big_data_route = Path(
    "../../logs/foundations/big_world/big_world/version_0/generation"
)
big_batch_paths = {}
for scope in [
    "london",
    "kr",
    "dutch",
    "uk",
    "dutch",
    "us",
    "chicago",
    "queensland",
    "victoria",
]:
    big_batch_paths[f"{scope}"] = big_data_route / f"{scope}"

big_schedules = {
    n: pd.read_csv(latest(p)[1] / "synthetic_schedules.csv")
    for n, p in big_batch_paths.items()
}
big_attributes = {
    n: load_attributes(latest(p)[1], default=None)
    for n, p in big_batch_paths.items()
}

Found versions for london: ['version_0']. Returning ../../logs/foundations/big_world/big_world/version_0/generation/london/version_0
Found versions for kr: ['version_0']. Returning ../../logs/foundations/big_world/big_world/version_0/generation/kr/version_0
Found versions for dutch: ['version_0']. Returning ../../logs/foundations/big_world/big_world/version_0/generation/dutch/version_0
Found versions for uk: ['version_0']. Returning ../../logs/foundations/big_world/big_world/version_0/generation/uk/version_0
Found versions for us: ['version_0']. Returning ../../logs/foundations/big_world/big_world/version_0/generation/us/version_0
Found versions for chicago: ['version_0']. Returning ../../logs/foundations/big_world/big_world/version_0/generation/chicago/version_0
Found versions for queensland: ['version_0']. Returning ../../logs/foundations/big_world/big_world/version_0/generation/queensland/version_0
Found versions for victoria: ['version_0']. Returning ../../logs/foundations/big_worl

In [50]:
selected = [
    "london",
    "kr",
    "dutch",
    "uk",
    "us",
    "chicago",
    "queensland",
    "victoria",
]

res_big = compare_many(
    schedules_a={k: training_schedules[k] for k in selected},
    attributes_a={k: training_attributes[k] for k in selected},
    schedules_b={k: big_schedules[k] for k in selected},
    attributes_b={k: big_attributes[k] for k in selected},
    split_on=[
        "source",
        "year",
        "age",
        "employment",
        "hh_income",
        "hh_zone",
        "day",
    ],
)

/home/fred/Projects/acteval/src/acteval/evaluate.py:799: UserWarning: combine(): row index differs from 'london' for source(s) ['kr', 'dutch', 'uk', 'us', 'chicago', 'queensland', 'victoria'] (e.g. different activities present in different targets). Aggregation weighting for non-first sources is approximate — see combine()'s docstring.
  return combine(results)


In [51]:
big_combined = combine({"base": res_base, "world": res_big})
big_combined.domains

/tmp/ipykernel_1500363/2287929800.py:1: UserWarning: combine(): row index differs from 'base' for source(s) ['world'] (e.g. different activities present in different targets). Aggregation weighting for non-first sources is approximate — see combine()'s docstring.
  big_combined = combine({"base": res_base, "world": res_big})
/home/fred/Projects/acteval/src/acteval/_result_frame.py:101: RuntimeWarning: The values in the array are unorderable. Pass `sort=False` to suppress this warning.
  combined_w = (self.weights[col] + target_weights) / 2
/home/fred/Projects/acteval/src/acteval/_result_frame.py:103: RuntimeWarning: The values in the array are unorderable. Pass `sort=False` to suppress this warning.
  weighted = (self.values[col].mul(combined_w)).groupby(groupby).sum()
/home/fred/Projects/acteval/src/acteval/_result_frame.py:101: RuntimeWarning: The values in the array are unorderable. Pass `sort=False` to suppress this warning.
  combined_w = (self.weights[col] + target_weights) / 2
/

ScheduleView [domains]  (.combined / .by_attribute / .by_category)

                base::london::london_base  base::kr::kr_base  base::dutch::dutch_base  base::uk::uk_base  base::us::us_base  base::chicago::chicago_base  base::queensland::queensland_base  base::victoria::victoria_base  world::london::london  world::kr::kr  world::dutch::dutch  world::uk::uk  world::us::us  world::chicago::chicago  world::queensland::queensland  world::victoria::victoria
domain                                                                                                                                                                                                                                                                                                                                                                                                
creativity                       0.221836           0.177756                 0.129836           0.229547           0.178740                     0.0903

In [54]:
results = pd.DataFrame()

for source in sources:
    old = big_combined.domains.combined.distances[
        f"base::{source}::{source}_base"
    ]
    new = big_combined.domains.combined.distances[f"world::{source}::{source}"]
    change = (new - old) / old
    results[source] = change

sizes_row = pd.Series(sizes, name="size").drop("training").to_frame().T
results = pd.concat([sizes_row, results], axis=0)


# sort cols by size
results = results[sizes_row.T.sort_values("size", ascending=False).index]

results = results.drop("creativity", axis=0)

results.columns = [
    "UK",
    "Korea",
    "Netherlands",
    "London",
    "Chicago",
    "Victoria",
    "US",
    "Queensland",
]

# manually rename index
results.index = [
    "Base sample size",
    "Feasibility",
    "Participation distribution",
    "Timing distribution",
    "Transitions distribution",
]

# reorder rows
results = results.loc[
    [
        "Base sample size",
        # "Feasibility",
        "Participation distribution",
        "Transitions distribution",
        "Timing distribution",
    ]
]


def styler(df):
    df.iloc[1:] = -df.iloc[1:]

    # subset that includes only the first row, formatted as integer
    size_subset = pd.IndexSlice[df.index[0], :]
    # subset that excludes the first row
    subset = pd.IndexSlice[df.index[1:], :]

    styled = (
        df.style.format("{:,.0f}", na_rep="", subset=size_subset)
        .background_gradient(cmap="RdYlGn", vmin=-1, vmax=1, subset=subset)
        .format("{:.1%}", na_rep="", subset=subset)
        .highlight_null("white")
        .set_table_styles(
            [
                {
                    "selector": "th, td",
                    "props": [
                        ("width", "75px"),
                        # ("min-width", "80px"),
                        # ("max-width", "80px"),
                        ("text-align", "center"),
                    ],
                },
                {
                    "selector": ".row_heading, .index_name",
                    "props": [("font-weight", "bold")],
                },
            ]
        )
    )
    return styled


average = (
    results.loc[
        [
            "Participation distribution",
            "Transitions distribution",
            "Timing distribution",
        ],
        :,
    ]
    .mean(axis=0)
    .to_frame(name="Average")
    .T
)

results = pd.concat([results, average], axis=0)
styler(results)

,UK,Korea,Netherlands,London,Chicago,Victoria,US,Queensland
Base sample size,"200,755","120,100","117,902","27,432","25,626","16,515","15,309","14,749"
Participation distribution,49.9%,-36.3%,30.2%,24.6%,32.7%,29.7%,17.2%,33.2%
Transitions distribution,18.8%,54.8%,25.3%,60.1%,61.0%,77.9%,76.5%,70.9%
Timing distribution,37.6%,-8.2%,30.7%,37.4%,44.6%,68.8%,72.3%,64.5%
Average,35.5%,3.4%,28.7%,40.7%,46.1%,58.8%,55.3%,56.2%


In [33]:
distances = pd.read_csv(data_route / "eval/groups/distances.csv").set_index(
    ["domain", "feature"]
)
distances[cols]

london_base    london   uk_base  \
domain         feature                                                    
creativity     conservatism                0.000066  0.000069  0.000010   
               homogeneity                 0.450022  0.468139  0.486588   
feasibility    consecutive                 0.007036  0.004812  0.013957   
               invalid                     0.007218  0.006015  0.014809   
               not home based              0.000182  0.001203  0.000917   
participations lengths                     0.525568  0.563297  0.493727   
               pair participation rate     0.010920  0.009600  0.008367   
               participation rate          0.108236  0.075194  0.067787   
timing         durations                   0.101978  0.044885  0.051278   
               joint-durations             0.157273  0.054399  0.068419   
               start times                 0.072273  0.017963  0.014612   
               start-durations             0.145675  0.101513  0.098412   
transitions    2-gram                      0.036721  0.013925  0.015244   
               3-gram                      0.003898  0.001735  0.001744   
               4-gram                      0.001096  0.000597  0.000498   

                                              uk  dutch_base     dutch  \
domain         feature                                                   
creativity     conservatism             0.000009    0.000012  0.000012   
               homogeneity              0.445115    0.274813  0.271938   
feasibility    consecutive              0.014625    0.027608  0.034597   
               invalid                  0.014968    0.028040  0.035055   
               not home based           0.000359    0.000466  0.000466   
participations lengths                  0.440039    0.307685  0.333088   
               pair participation rate  0.007456    0.006795  0.006395   
               participation rate       0.057307    0.062253  0.057067   
timing         durations                0.038184    0.033062  0.023933   
               joint-durations          0.060416    0.059248  0.033528   
               start times              0.011333    0.021101  0.012326   
               start-durations          0.075383    0.062189  0.054913   
transitions    2-gram                   0.010426    0.015047  0.010187   
               3-gram                   0.001397    0.001792  0.001221   
               4-gram                   0.000363    0.000506  0.000380   

                                         us_base        us  chicago_base  \
domain         feature                                                     
creativity     conservatism             0.000115  0.000116      0.000050   
               homogeneity              0.434058  0.439023      0.225201   
feasibility    consecutive              0.051473  0.009145      0.080816   
               invalid                  0.051473  0.009537      0.080855   
               not home based           0.000000  0.000392      0.000078   
participations lengths                  0.363870  0.336892      0.995577   
               pair participation rate  0.016196  0.006115      0.022762   
               participation rate       0.152673  0.046494      0.178234   
timing         durations                0.161602  0.039561      0.089862   
               joint-durations          0.255806  0.041820      0.139366   
               start times              0.134489  0.015135      0.059429   
               start-durations          0.193918  0.084371      0.179734   
transitions    2-gram                   0.051492  0.010466      0.036633   
               3-gram                   0.005187  0.001346      0.004545   
               4-gram                   0.001247  0.000486      0.001100   

                                         chicago  queensland_base  queensland  \
domain         feature                                                          
creativity     conservatism             0.000052  

In [34]:
results = pd.DataFrame()

for source in sources:
    old = distances[f"{source}_base"]
    new = distances[f"{source}"]
    change = (new - old) / old
    results[source] = change

sizes_row = pd.Series(sizes, name="size").drop("training").to_frame().T
results = pd.concat([sizes_row, results], axis=0)

# sort cols by size
results = results[sizes_row.T.sort_values("size", ascending=False).index]

styler(results)

,uk,kr,dutch,london,chicago,victoria,us,queensland
size,"200,755","120,100","117,902","27,432","25,626","16,515","15,309","14,749"
"('creativity', 'conservatism')",7.5%,-27.9%,0.4%,-3.4%,-3.4%,-3.0%,-0.9%,0.6%
"('creativity', 'homogeneity')",8.5%,-39.5%,1.0%,-4.0%,-11.5%,-6.7%,-1.1%,1.6%
"('feasibility', 'consecutive')",-4.8%,-50.3%,-25.3%,31.6%,67.2%,79.8%,82.2%,80.9%
"('feasibility', 'invalid')",-1.1%,-51.6%,-25.0%,16.7%,64.4%,79.0%,81.5%,79.7%
"('feasibility', 'not home based')",60.9%,-inf%,-0.0%,-560.0%,-3150.0%,-380.0%,-inf%,-inf%
"('participations', 'lengths')",10.9%,-131.2%,-8.3%,-7.2%,13.3%,27.2%,7.4%,25.6%
"('participations', 'pair participation rate')",10.9%,29.5%,5.9%,12.1%,40.6%,63.7%,62.2%,77.0%
"('participations', 'participation rate')",15.5%,-1.3%,8.3%,30.5%,30.3%,60.0%,69.5%,66.9%
"('timing', 'durations')",25.5%,-28.3%,27.6%,56.0%,54.5%,84.4%,75.5%,84.1%


In [ ]:
training_attributes["training"].hh_income.value_counts()

In [ ]:
def duration(schedules, act="work"):
    if "duration" not in schedules.columns:
        durations = schedules.end - schedules.start
    else:
        durations = schedules.duration
    filtered = durations[schedules.act == act]
    n = schedules.pid.nunique()
    return filtered.sum(0) / n


def count_trips(schedules):
    n = schedules.pid.nunique()
    return (len(schedules) - n) / n


def count_acts(schedules, act="work"):
    n = schedules.pid.nunique()
    return len(schedules.loc[schedules.act == act]) / n


def plot_conditionals(
    schedules, attributes, target_schedules, target_attributes, figsize=(8, 10)
):
    WORK_STATUS = "employment"
    INCOME = "hh_income"
    AGE = "age"

    work_order = ["student", "employed", "unemployed"]
    work_statuses = ["Student", "Employed", "Unemp."]

    age_order = ["≤18", "18-36", "36-51", "51-65", ">65"]
    ages = age_order

    incomes = ["≤21400", "21400-32526", "32526-47815", "47815-81570", ">81570"]
    colours = [
        "dodgerblue",
        "dodgerblue",
        "tomato",
        "tomato",
        "green",
        "green",
        "pink",
        "pink",
        "orange",
        "orange",
    ]
    styles = ["-", ":", "-", ":", "-", ":", "-", ":", "-", ":"]
    LW = 1.8

    fig, axs = plt.subplots(5, 3, figsize=figsize, sharex="col", sharey="row")

    # ==============================
    # trips employment
    splits = split_on(target_schedules, target_attributes, by=WORK_STATUS)
    data = {"Target UK": {}}
    for i, split in splits.items():
        data["Target UK"][i] = count_trips(split)

    for n, scheds in schedules.items():
        data[n] = {}
        atts = attributes[n]
        splits = split_on(scheds, atts, by=WORK_STATUS)
        for i, split in splits.items():
            data[n][i] = count_trips(split)

    df = pd.DataFrame(data)
    df = df.loc[work_order]
    ax = axs[0][0]
    # change to bar plot
    for n, col, s in zip(df.columns, colours, styles):
        ax.scatter(
            df.index,
            df[n],
            label=n,
            color=col,
            lw=LW,
            marker="_",
            s=1000,
            linestyle=s,
        )
    ax.set_ylabel("Trip Frequency")
    ax.legend().remove()

    # ==============================
    # trips income
    splits = split_on(target_schedules, target_attributes, by=INCOME)
    data = {"Target UK": {}}
    for i, split in splits.items():
        data["Target UK"][i] = count_trips(split)

    for n, scheds in schedules.items():
        data[n] = {}
        atts = attributes[n]
        splits = split_on(scheds, atts, by=INCOME)
        for i, split in splits.items():
            data[n][i] = count_trips(split)

    df = pd.DataFrame(data)
    df = df.loc[incomes]
    ax = axs[0][1]
    df.plot(ax=ax, color=colours, style=styles, lw=LW)
    ax.legend().remove()

    # ==================================
    # trips age group
    splits = split_on(target_schedules, target_attributes, by=AGE)
    data = {"Target UK": {}}
    for i, split in splits.items():
        data["Target UK"][i] = count_trips(split)

    for n, scheds in schedules.items():
        data[n] = {}
        atts = attributes[n]
        splits = split_on(scheds, atts, by=AGE)
        for i, split in splits.items():
            data[n][i] = count_trips(split)

    df = pd.DataFrame(data)
    ax = axs[0][2]
    df.loc[age_order].plot(ax=ax, color=colours, style=styles, lw=LW)
    ax.legend().remove()

    # =================================
    # work employment

    splits = split_on(target_schedules, target_attributes, by=WORK_STATUS)
    data = {"Target UK": {}}
    for i, split in splits.items():
        data["Target UK"][i] = count_acts(split)

    for n, scheds in schedules.items():
        data[n] = {}
        atts = attributes[n]
        splits = split_on(scheds, atts, by=WORK_STATUS)
        for i, split in splits.items():
            data[n][i] = count_acts(split)

    df = pd.DataFrame(data)
    df = df.loc[work_order]
    ax = axs[1][0]
    for n, col, s in zip(df.columns, colours, styles):
        ax.scatter(
            df.index,
            df[n],
            label=n,
            color=col,
            lw=LW,
            marker="_",
            s=1000,
            linestyle=s,
        )
    ax.set_ylabel("Working Frequency")
    ax.legend().remove()

    # =================================
    # work income

    splits = split_on(target_schedules, target_attributes, by=INCOME)
    data = {"Target UK": {}}
    for i, split in splits.items():
        data["Target UK"][i] = count_acts(split)

    for n, scheds in schedules.items():
        data[n] = {}
        atts = attributes[n]
        splits = split_on(scheds, atts, by=INCOME)
        for i, split in splits.items():
            data[n][i] = count_acts(split)

    df = pd.DataFrame(data)
    df = df.loc[incomes]
    ax = axs[1][1]
    df.plot(ax=ax, color=colours, style=styles, lw=LW)
    ax.legend().remove()

    # ====================================
    # work age group

    splits = split_on(target_schedules, target_attributes, by=AGE)
    data = {"Target UK": {}}
    for i, split in splits.items():
        data["Target UK"][i] = count_acts(split)

    for n, scheds in schedules.items():
        data[n] = {}
        atts = attributes[n]
        splits = split_on(scheds, atts, by=AGE)
        for i, split in splits.items():
            data[n][i] = count_acts(split)

    df = pd.DataFrame(data)
    ax = axs[1][2]
    df.loc[age_order].plot(ax=ax, color=colours, style=styles, lw=LW)
    ax.legend().remove()

    # =================================
    # shop employment

    splits = split_on(target_schedules, target_attributes, by=WORK_STATUS)
    data = {"Target UK": {}}
    for i, split in splits.items():
        data["Target UK"][i] = count_acts(split, act="shop")

    for n, scheds in schedules.items():
        data[n] = {}
        atts = attributes[n]
        splits = split_on(scheds, atts, by=WORK_STATUS)
        for i, split in splits.items():
            data[n][i] = count_acts(split, act="shop")

    df = pd.DataFrame(data)
    df = df.loc[work_order]
    ax = axs[2][0]
    for n, col, s in zip(df.columns, colours, styles):
        ax.scatter(
            df.index,
            df[n],
            label=n,
            color=col,
            lw=LW,
            marker="_",
            s=1000,
            linestyle=s,
        )
    ax.set_ylabel("Shopping Frequency")
    ax.legend().remove()

    # =================================
    # shop income

    splits = split_on(target_schedules, target_attributes, by=INCOME)
    data = {"Target UK": {}}
    for i, split in splits.items():
        data["Target UK"][i] = count_acts(split, act="shop")

    for n, scheds in schedules.items():
        data[n] = {}
        atts = attributes[n]
        splits = split_on(scheds, atts, by=INCOME)
        for i, split in splits.items():
            data[n][i] = count_acts(split, act="shop")

    df = pd.DataFrame(data)
    df = df.loc[incomes]
    ax = axs[2][1]
    df.plot(ax=ax, color=colours, style=styles, lw=LW)
    ax.legend().remove()

    # ====================================
    # shop age group

    splits = split_on(target_schedules, target_attributes, by=AGE)
    data = {"Target UK": {}}
    for i, split in splits.items():
        data["Target UK"][i] = count_acts(split, act="shop")

    for n, scheds in schedules.items():
        data[n] = {}
        atts = attributes[n]
        splits = split_on(scheds, atts, by=AGE)
        for i, split in splits.items():
            data[n][i] = count_acts(split, act="shop")

    df = pd.DataFrame(data)
    ax = axs[2][2]
    df.loc[age_order].plot(ax=ax, color=colours, style=styles, lw=LW)
    ax.legend().remove()

    # ====================================
    # work employment

    splits = split_on(target_schedules, target_attributes, by=WORK_STATUS)
    data = {"Target UK": {}}
    for i, split in splits.items():
        data["Target UK"][i] = duration(split)

    for n, scheds in schedules.items():
        data[n] = {}
        atts = attributes[n]
        splits = split_on(scheds, atts, by=WORK_STATUS)
        for i, split in splits.items():
            data[n][i] = duration(split)

    df = pd.DataFrame(data)
    df = df.loc[work_order]
    ax = axs[3][0]
    for n, col, s in zip(df.columns, colours, styles):
        ax.scatter(
            df.index,
            df[n],
            label=n,
            color=col,
            lw=LW,
            marker="_",
            s=1000,
            linestyle=s,
        )
    ax.legend().remove()
    ax.set_ylabel("Working Duration")

    # ====================================
    # work income

    splits = split_on(target_schedules, target_attributes, by=INCOME)
    data = {"Target UK": {}}
    for i, split in splits.items():
        data["Target UK"][i] = duration(split)

    for n, scheds in schedules.items():
        data[n] = {}
        atts = attributes[n]
        splits = split_on(scheds, atts, by=INCOME)
        for i, split in splits.items():
            data[n][i] = duration(split)

    df = pd.DataFrame(data)
    df = df.loc[incomes]
    ax = axs[3][1]
    df.plot(ax=ax, color=colours, style=styles, lw=LW)
    ax.legend().remove()

    # ====================================
    # work age group

    splits = split_on(target_schedules, target_attributes, by=AGE)
    data = {"Target UK": {}}
    for i, split in splits.items():
        data["Target UK"][i] = duration(split)

    for n, scheds in schedules.items():
        data[n] = {}
        atts = attributes[n]
        splits = split_on(scheds, atts, by=AGE)
        for i, split in splits.items():
            data[n][i] = duration(split)

    df = pd.DataFrame(data)
    ax = axs[3][2]
    df.loc[age_order].plot(ax=ax, color=colours, style=styles, lw=LW)
    ax.legend().remove()

    # ====================================
    # shop employment

    splits = split_on(target_schedules, target_attributes, by=WORK_STATUS)
    data = {"Target UK": {}}
    for i, split in splits.items():
        data["Target UK"][i] = duration(split, act="shop")

    for n, scheds in schedules.items():
        data[n] = {}
        atts = attributes[n]
        splits = split_on(scheds, atts, by=WORK_STATUS)
        for i, split in splits.items():
            data[n][i] = duration(split, act="shop")

    df = pd.DataFrame(data)
    df = df.loc[work_order]
    ax = axs[4][0]
    for n, col, s in zip(df.columns, colours, styles):
        ax.scatter(
            df.index,
            df[n],
            label=n,
            color=col,
            lw=LW,
            marker="_",
            s=1000,
            linestyle=s,
        )

    ax.legend().remove()
    ax.set_xlim(-0.5, 2.5)

    ax.set_ylabel("Shopping Duration")
    ax.set_xlabel("Employed")
    ax.set_xticks(list(range(len(work_order))), work_statuses, rotation=0)

    # ====================================
    # shop income

    splits = split_on(target_schedules, target_attributes, by=INCOME)
    data = {"Target UK": {}}
    for i, split in splits.items():
        data["Target UK"][i] = duration(split, act="shop")

    for n, scheds in schedules.items():
        data[n] = {}
        atts = attributes[n]
        splits = split_on(scheds, atts, by=INCOME)
        for i, split in splits.items():
            data[n][i] = duration(split, act="shop")

    df = pd.DataFrame(data)
    df = df.loc[incomes]
    ax = axs[4][1]
    df.plot(ax=ax, color=colours, style=styles, lw=LW)
    ax.legend().remove()
    ax.set_xlabel("Household Income")
    ax.set_xticks(
        [0, 1, 2, 3, 4],
        ["Lowest", "Low", "Mid", "High", "Highest"],
        rotation=90,
    )

    # ====================================
    # shop age group

    splits = split_on(target_schedules, target_attributes, by=AGE)
    data = {"Target UK": {}}
    for i, split in splits.items():
        data["Target UK"][i] = duration(split, act="shop")

    for n, scheds in schedules.items():
        data[n] = {}
        atts = attributes[n]
        splits = split_on(scheds, atts, by=AGE)
        for i, split in splits.items():
            data[n][i] = duration(split, act="shop")

    df = pd.DataFrame(data)
    ax = axs[4][2]
    df.loc[age_order].plot.line(ax=ax, color=colours, style=styles, lw=LW)
    ax.legend().remove()
    ax.set_xlabel("Age Group")
    ax.set_xticks(range(len(ages)), ages, rotation=90)

    # for ax in axs.flat:
    #     ax.grid(visible=True, which="major", axis="x", color="grey", linestyle="-")

    handles, labels = ax.get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="lower center",
        frameon=False,
        fontsize=10,
        bbox_to_anchor=(0.25, 1),
    )

    fig.align_ylabels(axs[:, 0])
    fig.align_xlabels(axs[-1, :])

    fig.tight_layout()


plot_conditionals(
    {
        "Synth UK": schedules["uk"],
        "Target Korea": training_schedules["kr"],
        "Synth Korea": schedules["kr"],
        "Target ND": training_schedules["dutch"],
        "Synth ND": schedules["dutch"],
    },
    {
        "Synth UK": attributes["uk"],
        "Target Korea": training_attributes["kr"],
        "Synth Korea": attributes["kr"],
        "Target ND": training_attributes["dutch"],
        "Synth ND": attributes["dutch"],
    },
    training_schedules["uk"],
    training_attributes["uk"],
)

In [ ]:
from acteval.features.times import durations_by_act, start_times_by_act
from acteval.plot.utils import _to_population
from matplotlib.figure import Axes, Figure


def time_plot(schedules, **kwargs) -> Figure:
    names = list(schedules.keys())
    pop = schedules[names[0]]
    acts = pop.act.value_counts().sort_values(ascending=False).index.tolist()
    acts.remove("home")
    n_acts = len(acts)
    rows = len(names)

    cmaps = kwargs.pop("cmaps", {})

    fig, axs = plt.subplots(
        rows,
        n_acts,
        figsize=kwargs.pop("figsize", (12, 5)),
        sharex=False,
        sharey=False,
    )

    for i, (name, y) in enumerate(schedules.items()):

        axs[i, 0].set_ylabel(name, fontsize=10, rotation=90)

        _joint_time_plot(_to_population(y), axs[i], acts, cmaps=cmaps)

    for ax in axs[-1]:
        ax.set_xlabel("Start times", fontsize=8)
        ax.set_xticks(
            [240, 480, 720, 960, 1200, 1440],
            labels=["04:00", "08:00", "12:00", "16:00", "20:00", "24:00"],
            rotation=90,
            fontsize=8,
        )

    for ax, title in zip(axs[:, 0], names):
        ax.text(
            -1,
            0.5,
            title,
            transform=ax.transAxes,
            fontsize=10,
            rotation=90,
            va="center",
            ha="center",
        )

    for ax, act in zip(axs[0], acts):
        ax.set_title(act.title(), fontsize=9)

    return fig


def _joint_time_plot(
    population,
    axs: Axes,
    acts: list[str],
    cmaps: dict,
    xmin: int = 240,
    xmax: int = 1441,
    ymin: int = 0,
    ymax: int = 960,
    xstep: int = 60,
    ystep: int = 60,
):
    pop = _to_population(population)
    start_feats = start_times_by_act(pop, bin_size=None)
    dur_feats = durations_by_act(pop, bin_size=None)

    start_bins = list(range(xmin, xmax, xstep))
    duration_bins = list(range(ymin, ymax, ystep))

    for i, act in enumerate(acts):
        cmap = cmaps.get(act)
        axs[i].set_xticks([])
        axs[i].set_yticks([])
        if act not in start_feats.data:
            continue
        act_starts = start_feats.data[act][0]
        act_durations = dur_feats.data[act][0]
        axs[i].hist2d(
            x=act_starts,
            y=act_durations,
            bins=(start_bins, duration_bins),
            cmap=cmap,
            cmin=1,
        )
    axs[0].set_ylabel("Durations", fontsize=8)
    axs[0].set_yticks(
        [0, 240, 480, 720, 960],
        labels=["00:00", "04:00", "08:00", "12:00", "16:00"],
        fontsize=8,
    )

In [ ]:
from acteval.plot.transitions import _probs_plot

cmaps = {
    "home": (0.55, 0.71, 0.82),  # #8CB6D1 steel blue
    "work": (0.88, 0.36, 0.32),  # #E15C52 red
    "education": (0.96, 0.78, 0.26),  # #F5C742 golden yellow
    "shop": (0.61, 0.49, 0.78),  # #9C7DC7 purple
    "visit": (0.94, 0.53, 0.23),  # #F0873B amber
    "escort": (0.29, 0.75, 0.73),  # #4ABFBA teal
    "leisure": (0.94, 0.53, 0.60),  # #F0879A rose
    "travel": (0.62, 0.62, 0.62),  # #9E9E9E mid grey
    "other": (0.71, 0.80, 0.48),  # #B5CC7A sage green
}

A = "uk"
B = "kr"

plot_schedules = {
    f"Target {A}": training_schedules[A],
    f"Synth {A}": schedules[A],
    f"Target {B}": training_schedules[B],
    f"Synth {B}": schedules[B],
}
plot_attributes = {
    f"Target {A}": training_attributes[A],
    f"Synth {A}": attributes[A],
    f"Target {B}": training_attributes[B],
    f"Synth {B}": attributes[B],
}

act_order = training_schedules["training"].act.value_counts().index.to_list()
class_map = {n: i for i, n in enumerate(act_order)}

FILTER = ["student", "employed", "unemployed", "retired"]
NAMES = ["Student", "Employed", "Unemployed", "Retired"]

h = len(plot_schedules)
w = len(FILTER)
fig, axes = plt.subplots(
    h, w + 2, figsize=(w * 2.2, h * 2), sharex=False, sharey=False
)

for i, (name, scheds) in enumerate(plot_schedules.items()):

    splits = split_on(scheds, plot_attributes[name], by="employment")

    for j, f in enumerate(FILTER):
        axes[0, j].set_title(NAMES[j], fontsize=12)
        selected = splits[f]
        ax = _probs_plot("", selected, cmap=cmaps, ax=axes[i, j])
        ax.set_title(None)
        ax.set_ylabel(None)
        ax.set_xlabel(None)
        ax.set_yticks([])
        ax.set_xticks([])

    axes[0, -2].set_title("Combined", fontsize=12)
    ax = _probs_plot("", scheds, cmap=cmaps, ax=axes[i, -2])
    ax.set_title(None)
    ax.set_ylabel(None)
    ax.set_xlabel(None)
    ax.set_yticks([])
    ax.set_xticks([])

    axes[i, 0].set_ylabel(name, fontsize=12)
    axes[i, 0].set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
    axes[i, 0].set_yticklabels(
        ["", "20% ", "40% ", "60% ", "80% ", "100%"], rotation=90, fontsize=8
    )

    axes[i, -1].axis("off")

elements = [Patch(facecolor=cmaps[act], label=act.title()) for act in act_order]
axes[0, -1].axis("off")
axes[0, -1].legend(handles=elements, loc="center left", frameon=False)


plt.subplots_adjust(wspace=0, hspace=0)

In [ ]:
selected = {
    "Target uk": split_on(
        training_schedules["uk"],
        training_attributes["uk"],
        by="employment",
        remove_unknown=True,
    )["employed"],
    "Synth uk": split_on(
        schedules["uk"], attributes["uk"], by="employment", remove_unknown=True
    )["employed"],
    "Target kr": split_on(
        training_schedules["kr"],
        training_attributes["kr"],
        by="employment",
        remove_unknown=True,
    )["employed"],
    "Synth kr": split_on(
        schedules["kr"], attributes["kr"], by="employment", remove_unknown=True
    )["employed"],
}


cmaps = {
    "work": "Reds",
    "education": "YlOrBr",
    "shop": "Purples",
    "visit": "Oranges",
    "escort": "GnBu",
    "leisure": "RdPu",
    "travel": "Greys",
    "other": "YlGn",
}

fig = time_plot(
    selected, figsize=(9, 7), observed_title="Target_UK", cmaps=cmaps
)
plt.subplots_adjust(wspace=0.2, hspace=0.2)

In [ ]:
freq_plotter(
    training_schedules["training"],
    training_attributes["training"],
    {
        "london_target": training_schedules["london"],
        "london": schedules["london"],
    },
    {
        "london_target": training_attributes["london"],
        "london": attributes["london"],
    },
    by="employment",
    filter=["student", "employed", "unemployed"],
)

In [ ]:
freq_plotter(
    training_schedules["training"],
    training_attributes["training"],
    {"uk_target": training_schedules["uk"], "uk": schedules["uk"]},
    {"uk_target": training_attributes["uk"], "uk": attributes["uk"]},
    by="employment",
    filter=["student", "employed", "unemployed"],
)

In [ ]:
freq_plotter(
    training_schedules["training"],
    training_attributes["training"],
    {"us_target": training_schedules["us"], "us": schedules["us"]},
    {"us_target": training_attributes["us"], "us": attributes["us"]},
    by="employment",
    filter=["student", "employed", "unemployed"],
)

In [ ]:
freq_plotter(
    training_schedules["training"],
    training_attributes["training"],
    {
        "london_target": training_schedules["london"],
        "london": schedules["london"],
    },
    {
        "london_target": training_attributes["london"],
        "london": attributes["london"],
    },
    by="age",
    filter=["≤18", "18-36", "36-51", "51-65", ">65"],
)

In [ ]:
freq_plotter(
    training_schedules["training"],
    training_attributes["training"],
    {"uk_target": training_schedules["uk"], "uk": schedules["uk"]},
    {"uk_target": training_attributes["uk"], "uk": attributes["uk"]},
    by="age",
    filter=["≤18", "18-36", "36-51", "51-65", ">65"],
)

In [ ]:
freq_plotter(
    training_schedules["training"],
    training_attributes["training"],
    {"us_target": training_schedules["us"], "us": schedules["us"]},
    {"us_target": training_attributes["us"], "us": attributes["us"]},
    by="age",
    filter=["≤18", "18-36", "36-51", "51-65", ">65"],
)

In [ ]:
freq_plotter(
    training_schedules["training"],
    training_attributes["training"],
    {"kr_target": training_schedules["kr"], "kr": schedules["kr"]},
    {"kr_target": training_attributes["kr"], "kr": attributes["kr"]},
    by="age",
    filter=["≤18", "18-36", "36-51", "51-65", ">65"],
)

In [ ]:
f = joint_time_distributions_plot(
    observed=training_schedules["dutch"],
    ys={"dutch": schedules["dutch"]},
    cmap=cmap,
    figsize=(12, 4),
)

f = joint_time_distributions_plot(
    observed=training_schedules["kr"],
    ys={"kr": schedules["kr"]},
    cmap=cmap,
    figsize=(12, 4),
)
# f = frequency_plots(
#     observed=target, ys=training_schedules, cmap=cmap, figsize=(12, 6)
# )
# f = sequence_prob_plot(
#     observed=target, ys=training_schedules, cmap=cmap, figsize=(12, 6)
# )

In [ ]:
# training_attributes = {
#     "uk": pd.read_csv(training_data_route / "uk_attributes.csv"),
#     "uk+aus": pd.read_csv(training_data_route / "uk+aus_attributes.csv"),
#     "western": pd.read_csv(training_data_route / "west_attributes.csv"),
#     "world": pd.read_csv(training_data_route / "world_attributes.csv"),
# }

# training_schedules = {
#     "uk": pd.read_csv(training_data_route / "uk_activities.csv"),
#     "uk+aus": pd.read_csv(training_data_route / "uk+aus_activities.csv"),
#     "western": pd.read_csv(training_data_route / "west_activities.csv"),
#     "world": pd.read_csv(training_data_route / "world_activities.csv"),
# }

In [ ]:
counter = {}
# load each and get counts per source
counter["baseline"] = target_attributes.source.value_counts()
for name, data in training_attributes.items():
    print(data.source.unique())
    counter[name] = data.source.value_counts()
df = pd.DataFrame(counter)


order = ["ktdb", "ltds", "vista", "qhts", "nhts", "nts"]
df = df.loc[order].fillna(0)
totals = df.sum()
totals_row = pd.DataFrame([totals], index=["Total"])
df = pd.concat([df, totals_row])
print(df.to_latex(float_format="{:.0f}".format))

In [ ]:
def distance_table(batch_path, suffix=""):
    df = pd.read_csv(batch_path / f"domain_distances{suffix}.csv").set_index(
        "domain"
    )
    order = [
        "participations",
        "transitions",
        "timing",
        "feasibility",
        "creativity",
    ]
    print(df.loc[order].to_latex(float_format="{:.3f}".format))
    return df.loc[order]


df = distance_table(Path("../../logs/foundations/B"), suffix="")

df

In [ ]:
df_alt = distance_table(Path("../../logs/foundations/B_alt"), suffix="")
diff = 100 * (df - df_alt) / df_alt
order = ["participations", "transitions", "timing", "feasibility", "creativity"]
print(diff.loc[order].to_latex(float_format="{:.1f}\%".format))
diff

In [ ]:
# _ = plot_b(
#     training_schedules, training_attributes, target_schedules, target_attributes
# )

In [ ]:
batch_root = Path("../../logs/foundations/B")

batch_paths = {
    "B0 (baseline)": batch_root / "baseline",
    "B1 (UK)": batch_root / "uk",
    "B2 (UK+AUS)": batch_root / "uk+aus",
    "B3 (West)": batch_root / "west",
    "B4 (World)": batch_root / "world",
}

# find data
data_root = Path("../../tmp/foundation")
target_schedules = pd.read_csv(
    data_root / "experiment_b" / "nts_activities_2024.csv"
)
acts = list(target_schedules.act.value_counts(ascending=False).index)
class_map = {n: i for i, n in enumerate(acts)}

target_attributes = pd.read_csv(
    data_root / "experiment_b" / "nts_attributes_2024.csv"
)

schedules = {
    n: pd.read_csv(latest(p)[1] / "synthetic_schedules.csv")
    for n, p in batch_paths.items()
}
attributes = {
    n: load_attributes(latest(p)[1], default=target_attributes)
    for n, p in batch_paths.items()
}
print(schedules.keys())
print(attributes.keys())

In [ ]:
_ = plot_b(schedules, attributes, target_schedules, target_attributes)